In [1]:
import os
import sys

project_root = "/root/work/tvm-ansor"
os.environ["TVM_HOME"] = f"{project_root}"
os.environ["TVM_LIBRARY_PATH"] = f"{project_root}/build-release"
if f"{project_root}/python" not in sys.path:
    sys.path.insert(0, f"{project_root}/python")

sys.path = [p for p in sys.path if not p.startswith(f"{project_root}/build")]
sys.path.append(f"{project_root}/build-release")
os.environ["LD_LIBRARY_PATH"] = f"{project_root}/build-release:" + os.environ.get("LD_LIBRARY_PATH", "")


import numpy as np
import tvm
from tvm import auto_scheduler
from tvm.auto_scheduler import SketchPolicy

TARGET = tvm.target.Target("cuda")



In [2]:
from modules.common import load_and_register_tasks
tasks = load_and_register_tasks()
# tasks_by_wkey = {task.workload_key: task for task in tasks}
concrete_states = {}
for idx, task in enumerate(tasks):
    concrete_state = SketchPolicy(task, params={'sample_init_no_invalid': 1 }, verbose=False).generate_concrete_sketches()
    for i, state in enumerate(concrete_state):
        concrete_states[f"{task.workload_key}_{i}"] = (task, state)

In [9]:
concrete_states

{'["1aa729c96f4afc0cf6bf84dff07364c6", [1, 18, 9, 1, 512], [1, 1, 1, 1, 512]]_0': (auto_scheduler.SearchTask(0x1ae34510),
  Placeholder: p0
  blockIdx.x ax0@ax1@ax2@ax3@ax4@.0 (0,16)
    threadIdx.x ax0@ax1@ax2@ax3@ax4@.1 (0,32)
      for rv0 (0,18)
        for rv1 (0,9)
          for rv2 (0,1)
            adaptive_pool_sum = ...
  blockIdx.x ax0@ax1@ax2@ax3@ax4@.0 (0,16)
    threadIdx.x ax0@ax1@ax2@ax3@ax4@.1 (0,32)
      adaptive_pool_avg = ...),
 '["25252ef28760d56401943904a46661f3", [1, 16, 16, 480], [1, 1, 1, 480]]_0': (auto_scheduler.SearchTask(0x1ae5a990),
  Placeholder: p0
  adaptive_pool_sum auto_unroll: 16
  blockIdx.x ax0@ax1@ax2@ax3@.0 (0,15)
    threadIdx.x ax0@ax1@ax2@ax3@.1 (0,32)
      for rv0 (0,16)
        for rv1 (0,16)
          adaptive_pool_sum = ...
  blockIdx.x ax0@ax1@ax2@ax3@.0 (0,15)
    threadIdx.x ax0@ax1@ax2@ax3@.1 (0,32)
      adaptive_pool_avg = ...),
 '["25252ef28760d56401943904a46661f3", [1, 16, 16, 672], [1, 1, 1, 672]]_0': (auto_scheduler.SearchTask(

In [11]:
from modules.param_manager import build_symbolic_state
sketch_idx = 600
sample = list(concrete_states.values())[sketch_idx]
sym_state = build_symbolic_state(sample[0], sample[1])
sym_state

Placeholder: p0, p1, p2
blockIdx.x ax0.0@ax1.0@ax2.0@ax3.0@ (0,ceil(ceil(ceil(7/(min(sp_2_2*sp_2_3,7)))/(min(sp_2_1,ceil(7/(min(sp_2_2*sp_2_3,7))))))/(min(sp_2_0,ceil(ceil(7/(min(sp_2_2*sp_2_3,7)))/(min(sp_2_1,ceil(7/(min(sp_2_2*sp_2_3,7)))))))))*ceil(ceil(ceil(7/(min(sp_3_2*sp_3_3,7)))/(min(sp_3_1,ceil(7/(min(sp_3_2*sp_3_3,7))))))/(min(sp_3_0,ceil(ceil(7/(min(sp_3_2*sp_3_3,7)))/(min(sp_3_1,ceil(7/(min(sp_3_2*sp_3_3,7)))))))))*ceil(ceil(ceil(512/(min(sp_4_2*sp_4_3,512)))/(min(sp_4_1,ceil(512/(min(sp_4_2*sp_4_3,512))))))/(min(sp_4_0,ceil(ceil(512/(min(sp_4_2*sp_4_3,512)))/(min(sp_4_1,ceil(512/(min(sp_4_2*sp_4_3,512))))))))))
  vthread ax0.1@ax1.1@ax2.1@ax3.1@ (0,min(sp_2_0,ceil(ceil(7/(min(sp_2_2*sp_2_3,7)))/(min(sp_2_1,ceil(7/(min(sp_2_2*sp_2_3,7)))))))*min(sp_3_0,ceil(ceil(7/(min(sp_3_2*sp_3_3,7)))/(min(sp_3_1,ceil(7/(min(sp_3_2*sp_3_3,7)))))))*min(sp_4_0,ceil(ceil(512/(min(sp_4_2*sp_4_3,512)))/(min(sp_4_1,ceil(512/(min(sp_4_2*sp_4_3,512))))))))
    threadIdx.x ax0.2@ax1.2@ax2.2@ax3.2

In [5]:
from modules.schedule_generator import ScheduleGenerator
sg = ScheduleGenerator(sym_state, task=sample[0], base_state=sample[1])
import random
rng = random.Random(0)
params = sg.randomize_params(rng=rng, max_retries=1)

In [7]:
sg

In [6]:
params

{'sp_4_1': 72,
 'sp_3_1': 5,
 'sp_2_1': 2,
 'sp_1_1': 1,
 'sp_1_0': 1,
 'sp_1_2': 1,
 'sp_1_3': 1,
 'sp_2_0': 1,
 'sp_2_2': 3,
 'sp_2_3': 1,
 'sp_3_0': 1,
 'sp_3_2': 1,
 'sp_3_3': 1,
 'sp_4_0': 1,
 'sp_4_2': 1,
 'sp_4_3': 1,
 'sp_5_0': 1,
 'sp_5_1': 3,
 'sp_6_0': 1,
 'sp_6_1': 1,
 'sp_26_0': 3,
 'sp_31_0': 2,
 'ur_35': 1024}